<a href="https://colab.research.google.com/github/allengr220/cognitive-ledger/blob/main/longevity/nhanes_extended_dataset_builderV3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# NHANES Extended Dataset Builder V3 — Frozen Spec

## Purpose
Build the extended NHANES analytic dataset used for PhysioRisk modeling and related survival analyses.

This notebook has two distinct outputs:

1. **All-cycle master dataset**
   - Builds a harmonized NHANES biomarker cohort across available continuous cycles.
   - Useful for feature coverage, harmonization checks, and non-survival exploratory work.

2. **Extended survival modeling dataset**
   - Builds the mortality-linked analytic cohort used for PhysioRisk and survival modeling.
   - Restricted to cycles with usable linked mortality follow-up.

---

## Frozen design decisions

### 1. Survival analysis window
The mortality-linked survival dataset is restricted to:

- **1999–2000**
- **2001–2002**
- **2003–2004**
- **2005–2006**
- **2007–2008**
- **2009–2010**
- **2011–2012**
- **2013–2014**
- **2015–2016**

### 2. Excluded cycles
- **2017–2018 is excluded from the survival dataset**
  - reason: no linked mortality coverage in `nhanes_mortality_2019_verified.parquet`
  - consequence: these rows cannot be assigned valid `PERMTH_INT` and `MORTSTAT`

- **Pre-1999 NHANES is excluded**
  - reason: inconsistent survey structure, biomarker availability, and harmonization burden

### 3. Mortality linkage rules
Mortality linkage is performed using:

- `SEQN` as the unique participant identifier
- one-to-one merge validation
- explicit post-merge coverage audit by `cycle_start_year`

### 4. Survival dataset invariants
All rows retained in the final mortality-linked modeling dataset must satisfy:

- non-null `SEQN`
- non-null `cycle_start_year`
- non-null `PERMTH_INT`
- non-null `MORTSTAT`
- one row per participant

If any included cycle has incomplete mortality coverage, the notebook should fail loudly rather than proceed silently.

---

## Primary source files

### NHANES source data
Continuous NHANES cycle files used to assemble the harmonized cohort.

### Mortality linkage file
- `/content/drive/MyDrive/LRE/data/pub/nhanes_mortality_2019_verified.parquet`

This mortality file provides usable coverage through **2015–2016**, but not **2017–2018**.

---

## Key variables expected in final survival dataset

### identifiers / design
- `SEQN`
- `cycle_start_year`

### survival outcome
- `MORTSTAT`
- `PERMTH_INT`
- optionally `PERMTH_EXM`
- optionally `ELIGSTAT`

### baseline demographics
- `RIDAGEYR`
- `RIAGENDR`

### biomarker panel
Expected to include the harmonized PhysioRisk candidate biomarker set where available, including core chemistry, hematology, anthropometric, blood pressure, and lipid variables.

---

## Outputs

### Master harmonized dataset
A broad harmonized NHANES dataset spanning all loaded continuous cycles.

### Final mortality-linked modeling dataset
- **`nhanes_1999_2016_extended_mortality_model_dataset.parquet`**

This is the frozen extended survival dataset intended for downstream PhysioRisk modeling and benchmarking.

---

## Operational intent
This notebook is a **data-construction notebook**, not a modeling notebook.

Its job is to:
- harmonize cycle-level NHANES inputs
- construct `cycle_start_year`
- merge validated mortality linkage
- enforce cohort eligibility rules
- export a reproducible mortality-linked dataset for downstream analysis

Any modeling notebook consuming this output should be able to assume that the mortality-linked cohort has already passed dataset integrity checks.

---

## Failure policy
The notebook should stop if any of the following occur:

- duplicate `SEQN` after merge
- missing `PERMTH_INT` in retained survival rows
- missing `MORTSTAT` in retained survival rows
- unexpected cycle present in final survival dataset
- mortality merge produces incomplete coverage for an allowed cycle

Silent partial inclusion is not allowed.

---

## Rationale
The analytic window is defined by the intersection of:

1. continuous NHANES comparability
2. harmonizable biomarker measurement structure
3. available linked mortality follow-up

For this project, that valid window is **1999–2016**.

In [ ]:
!pip -q install pyreadstat

import pandas as pd
import numpy as np
import requests
import pyreadstat
from io import BytesIO

def xpt_url(folder: str, filename: str) -> str:
    return f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{folder}/DataFiles/{filename}"

def read_xpt(url):
    r = requests.get(url)
    r.raise_for_status()
    return pd.read_sas(BytesIO(r.content), format="xport")

def first_present(df: pd.DataFrame, candidates, required=False):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of these columns found: {candidates}")
    return None

In [ ]:
ID_COL = "SEQN"
TIME_COL = "PERMTH_INT"
EVENT_COL = "MORTSTAT"
AGE_COL = "RIDAGEYR"
SEX_COL = "RIAGENDR"
RACE_COL = "RIDRETH3"
CYCLE_COL = "CYCLE"

VALID_SURVIVAL_CYCLES = [
    "1999-2000", "2001-2002", "2003-2004",
    "2005-2006", "2007-2008", "2009-2010",
    "2011-2012", "2013-2014", "2015-2016",
]

def cycle_start_year_from_label(label: str) -> int:
    return int(str(label).split("-")[0])


PUB_DIR = "/content/drive/MyDrive/LRE/data/pub"
BIOMARKER_MASTER_PATH = f"{PUB_DIR}/nhanes_biomarker_master_verified.parquet"
MORTALITY_PATH = f"{PUB_DIR}/nhanes_mortality_2019_verified.parquet"
SURVIVAL_MASTER_PATH = f"{PUB_DIR}/nhanes_1999_2016_survival_master.parquet"
EXTENDED_MODEL_PATH = f"{PUB_DIR}/nhanes_1999_2016_extended_mortality_model_dataset.parquet"

SURVIVAL_CYCLES = [c for c in CYCLES] if "CYCLES" in globals() else None


In [ ]:
CYCLES = [
    {"label": "1999-2000", "folder": "1999", "suffix": "",   "mort": "NHANES_1999_2000_MORT_2019_PUBLIC.dat"},
    {"label": "2001-2002", "folder": "2001", "suffix": "_B", "mort": "NHANES_2001_2002_MORT_2019_PUBLIC.dat"},
    {"label": "2003-2004", "folder": "2003", "suffix": "_C", "mort": "NHANES_2003_2004_MORT_2019_PUBLIC.dat"},
    {"label": "2005-2006", "folder": "2005", "suffix": "_D", "mort": "NHANES_2005_2006_MORT_2019_PUBLIC.dat"},
    {"label": "2007-2008", "folder": "2007", "suffix": "_E", "mort": "NHANES_2007_2008_MORT_2019_PUBLIC.dat"},
    {"label": "2009-2010", "folder": "2009", "suffix": "_F", "mort": "NHANES_2009_2010_MORT_2019_PUBLIC.dat"},
    {"label": "2011-2012", "folder": "2011", "suffix": "_G", "mort": "NHANES_2011_2012_MORT_2019_PUBLIC.dat"},
    {"label": "2013-2014", "folder": "2013", "suffix": "_H", "mort": "NHANES_2013_2014_MORT_2019_PUBLIC.dat"},
    {"label": "2015-2016", "folder": "2015", "suffix": "_I", "mort": "NHANES_2015_2016_MORT_2019_PUBLIC.dat"},
    {"label": "2017-2018", "folder": "2017", "suffix": "_J", "mort": "NHANES_2017_2018_MORT_2019_PUBLIC.dat"},
]

SURVIVAL_CYCLES = [c for c in CYCLES if c["label"] in VALID_SURVIVAL_CYCLES]
EXCLUDED_SURVIVAL_CYCLES = [c["label"] for c in CYCLES if c["label"] not in VALID_SURVIVAL_CYCLES]

print("All NHANES cycles loaded for biomarker master:", [c["label"] for c in CYCLES])
print("Cycles used for survival datasets:", [c["label"] for c in SURVIVAL_CYCLES])
print("Excluded from survival datasets:", EXCLUDED_SURVIVAL_CYCLES)


SURVIVAL_CYCLES = [c for c in CYCLES if c["label"] in VALID_SURVIVAL_CYCLES]
print("Survival cycles:", [c["label"] for c in SURVIVAL_CYCLES])


In [ ]:
def try_read_any(folder, filenames, required=True):
    last_err = None
    for fn in filenames:
        try:
            return read_xpt(xpt_url(folder, fn)), fn
        except Exception as e:
            last_err = e
    if required:
        raise last_err
    return None, None


def select_and_rename(df, candidates_map, seqn_required=True):
    if df is None:
        return None

    cols = []
    new_names = []

    if seqn_required:
        seqn_col = first_present(df, ["SEQN"], required=True)
        cols.append(seqn_col)
        new_names.append("SEQN")

    for new_col, candidates in candidates_map.items():
        found = first_present(df, candidates, required=False)
        if found is not None:
            cols.append(found)
            new_names.append(new_col)

    out = df[cols].copy()
    out.columns = new_names
    return out


def load_cycle(cycle: dict) -> pd.DataFrame:
    """
    Robust NHANES cycle loader for PhysioRisk / extended biomarker work.

    Returns a per-cycle dataframe with:
      - identifiers / demographics
      - survey design variables
      - biomarker panel (14 + optional extras for 17)
      - cycle metadata

    Notes:
      - CRP is loaded into LBXHSCRP (raw as found in source file)
      - CRP is also copied into LBXHSCRP_STD as a placeholder for later
        verified unit harmonization
      - mortality is not merged here; do that downstream on SEQN
    """
    suffix = cycle["suffix"]
    folder = cycle["folder"]

    # -------------------------
    # Read source files
    # -------------------------
    demo, demo_fn = try_read_any(folder, [f"DEMO{suffix}.XPT"])
    bmx,  bmx_fn  = try_read_any(folder, [f"BMX{suffix}.XPT"])
    bpx,  bpx_fn  = try_read_any(folder, [f"BPX{suffix}.XPT"])

    ghb, ghb_fn = try_read_any(folder, [
        f"GHB{suffix}.XPT",
        f"L10{suffix}.XPT",
        "LAB10.XPT",
    ])

    hdl, hdl_fn = try_read_any(folder, [
        f"HDL{suffix}.XPT",
        f"L13{suffix}.XPT",
        "LAB13.XPT",
    ])

    tchol, tchol_fn = try_read_any(folder, [
        f"TCHOL{suffix}.XPT",
        f"L13{suffix}.XPT",
        "LAB13.XPT",
    ])

    trig, trig_fn = try_read_any(folder, [
        f"TRIGLY{suffix}.XPT",
        f"L13AM{suffix}.XPT",
        "L13AM.XPT",
        "LAB13AM.XPT",
    ])

    crp, crp_fn = try_read_any(folder, [
        # Primary hs-CRP naming
        f"HSCRP{suffix}.XPT",

        # Alternate naming seen in some cycles
        f"CRP{suffix}.XPT",

        # Older lab naming
        f"L11{suffix}.XPT",
        "LAB11.XPT",

        # 👇 ADD THESE (critical)
        f"HSCRP_{suffix.strip('_')}.XPT",   # handles weird underscore cases
        f"CRP_{suffix.strip('_')}.XPT",

        # fallback broad patterns (if your loader supports it)
    ], required=False)

    biochem, biochem_fn = try_read_any(folder, [
        f"BIOPRO{suffix}.XPT",
        f"L40{suffix}.XPT",
        "LAB18.XPT",
        "L40_B.XPT",
        "L40_C.XPT",
        "BIOPRO_D.XPT",
        "BIOPRO_E.XPT",
        "BIOPRO_F.XPT",
        "BIOPRO_G.XPT",
        "BIOPRO_H.XPT",
        "BIOPRO_I.XPT",
        "BIOPRO_J.XPT",
    ], required=False)

    # Glucose / fasting glucose coverage
    glu, glu_fn = try_read_any(folder, [
        f"GLU{suffix}.XPT",
        f"GLU_F{suffix}.XPT",
        f"LAB10AM{suffix}.XPT",
        "LAB10AM.XPT",
        f"L40{suffix}.XPT",
        "LAB40.XPT",
    ], required=False)

    # CBC coverage, including older L25 files
    cbc, cbc_fn = try_read_any(folder, [
        f"CBC{suffix}.XPT",
        f"L25{suffix}.XPT",
        "LAB25.XPT",
    ], required=False)

    # -------------------------
    # Demo / survey structure
    # -------------------------
    demo = demo[[
        first_present(demo, ["SEQN"], required=True),
        first_present(demo, ["RIDAGEYR"], required=True),
        first_present(demo, ["RIAGENDR"], required=True),
        first_present(demo, ["RIDRETH3", "RIDRETH1"], required=True),
        first_present(demo, ["SDDSRVYR"], required=False),
        first_present(demo, ["WTMEC2YR"], required=False),
        first_present(demo, ["SDMVPSU"], required=False),
        first_present(demo, ["SDMVSTRA"], required=False),
    ]].copy()

    demo.columns = [
        "SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH3",
        "SDDSRVYR", "WTMEC2YR", "SDMVPSU", "SDMVSTRA"
    ]

    bmx = bmx[[
        first_present(bmx, ["SEQN"], required=True),
        first_present(bmx, ["BMXBMI"], required=True),
        first_present(bmx, ["BMXWAIST"], required=True),
    ]].copy()
    bmx.columns = ["SEQN", "BMXBMI", "BMXWAIST"]

    bpx = bpx[[
        first_present(bpx, ["SEQN"], required=True),
        first_present(bpx, ["BPXSY1"], required=True),
        first_present(bpx, ["BPXDI1"], required=True),
    ]].copy()
    bpx.columns = ["SEQN", "BPXSY1", "BPXDI1"]

    ghb = ghb[[
        first_present(ghb, ["SEQN"], required=True),
        first_present(ghb, ["LBXGH", "LB2GH"], required=True),
    ]].copy()
    ghb.columns = ["SEQN", "LBXGH"]

    hdl = hdl[[
        first_present(hdl, ["SEQN"], required=True),
        first_present(hdl, ["LBDHDD", "LBXHDD", "LBDHDL", "LBXHDDSI"], required=True),
    ]].copy()
    hdl.columns = ["SEQN", "LBDHDD"]

    tchol = tchol[[
        first_present(tchol, ["SEQN"], required=True),
        first_present(tchol, ["LBXTC", "LB2TCH", "LB2SCH"], required=True),
    ]].copy()
    tchol.columns = ["SEQN", "LBXTC"]

    trig = trig[[
        first_present(trig, ["SEQN"], required=True),
        first_present(trig, ["LBXTR", "LB2TR", "LBDTRSI"], required=True),
    ]].copy()
    trig.columns = ["SEQN", "LBXTR"]

    # -------------------------
    # Merge core blocks
    # -------------------------
    out = (
        demo.merge(bmx, on="SEQN", how="left")
            .merge(bpx, on="SEQN", how="left")
            .merge(ghb, on="SEQN", how="left")
            .merge(hdl, on="SEQN", how="left")
            .merge(tchol, on="SEQN", how="left")
            .merge(trig, on="SEQN", how="left")
    )

    # -------------------------
    # CRP
    # -------------------------
    if crp is not None:
        crp = crp[[
            first_present(crp, ["SEQN"], required=True),
            first_present(crp, ["LBXHSCRP", "LBXCRP"], required=True),
        ]].copy()
        crp.columns = ["SEQN", "LBXHSCRP"]
        out = out.merge(crp, on="SEQN", how="left")
    else:
        out["LBXHSCRP"] = np.nan

    # Preserve raw CRP and create a verified-standardization placeholder
    out["LBXHSCRP_STD"] = out["LBXHSCRP"]

        # -------------------------
    # Chemistry block
    # -------------------------
    # We want SERUM albumin, SERUM creatinine, and SERUM alkaline phosphatase.
    # Older cycles (2001-2004) use L40_B / L40_C.
    # 1999-2000 uses LAB18.
    # 2005+ generally uses BIOPRO_*.

    biochem, biochem_fn = try_read_any(folder, [
        # Modern standard biochemistry profile
        f"BIOPRO{suffix}.XPT",

        # Older standard biochemistry profile (critical for 2001-2004)
        f"L40{suffix}.XPT",

        # 1999-2000 fallback
        "LAB18.XPT",

        # Explicit fallbacks
        "L40_B.XPT",
        "L40_C.XPT",
        "BIOPRO_D.XPT",
        "BIOPRO_E.XPT",
        "BIOPRO_F.XPT",
        "BIOPRO_G.XPT",
        "BIOPRO_H.XPT",
        "BIOPRO_I.XPT",
        "BIOPRO_J.XPT",
    ], required=False)

    if biochem is not None:
        biochem = select_and_rename(
            biochem,
            {
                "LBXSAL":   ["LBXSAL", "LBDSALSI", "LB2SAL", "LB2AL"],
                "LBXSCR":   ["LBXSCR", "LBDSCRSI", "LB2SCR"],
                "LBXSAPSI": ["LBXSAPSI", "LBDSAPSI", "LB2SAPSI", "LB2AP"],
            }
        )
        out = out.merge(biochem, on="SEQN", how="left")
    else:
        out["LBXSAL"] = np.nan
        out["LBXSCR"] = np.nan
        out["LBXSAPSI"] = np.nan

    # -------------------------
    # Glucose block
    # -------------------------
    if glu is not None:
        glu = select_and_rename(
            glu,
            {
                "LBXGLU": ["LBXGLU", "LBDGLUSI", "LBDSGLSI", "LB2GLU"],
            }
        )
        out = out.merge(glu, on="SEQN", how="left")
    else:
        out["LBXGLU"] = np.nan

    # -------------------------
    # CBC block
    # -------------------------
    if cbc is not None:
        cbc = select_and_rename(
            cbc,
            {
                "LBXWBCSI": ["LBXWBCSI", "LBDWBCSI", "LBXWBC"],
                "LBXLYPCT": ["LBXLYPCT", "LBDLYMNO", "LBXLYP"],
                "LBXMCV":   ["LBXMCV", "LBDMCVSI", "LBXMCVSI"],
                "LBXRDW":   ["LBXRDW", "LBDRDW", "LBXRDWSI"],
            }
        )
        out = out.merge(cbc, on="SEQN", how="left")
    else:
        out["LBXWBCSI"] = np.nan
        out["LBXLYPCT"] = np.nan
        out["LBXMCV"] = np.nan
        out["LBXRDW"] = np.nan

    # -------------------------
    # Type coercion
    # -------------------------
    out["SEQN"] = pd.to_numeric(out["SEQN"], errors="coerce").astype("Int64")

    numeric_cols = [
        "RIDAGEYR", "RIAGENDR", "RIDRETH3",
        "SDDSRVYR", "WTMEC2YR", "SDMVPSU", "SDMVSTRA",
        "BMXBMI", "BMXWAIST", "BPXSY1", "BPXDI1",
        "LBXGH", "LBDHDD", "LBXTC", "LBXTR",
        "LBXHSCRP", "LBXHSCRP_STD",
        "LBXSAL", "LBXSCR", "LBXSAPSI",
        "LBXWBCSI", "LBXLYPCT",
        "LBXGLU", "LBXMCV", "LBXRDW",
    ]
    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    # -------------------------
    # Metadata
    # -------------------------
    out["CYCLE"] = cycle["label"]
    out["CYCLE_SUFFIX"] = suffix
    out["CYCLE_FOLDER"] = folder

    # -------------------------
    # Diagnostics
    # -------------------------
    diag_cols = [
        "RIDAGEYR", "RIAGENDR", "RIDRETH3",
        "BMXBMI", "BMXWAIST", "BPXSY1", "BPXDI1",
        "LBXGH", "LBDHDD", "LBXTC", "LBXTR",
        "LBXHSCRP", "LBXSAL", "LBXSCR", "LBXSAPSI",
        "LBXWBCSI", "LBXLYPCT",
        "LBXGLU", "LBXMCV", "LBXRDW"
    ]
    nonnull_summary = {c: int(out[c].notna().sum()) for c in diag_cols if c in out.columns}

    print(
        f"{cycle['label']} files -> "
        f"demo={demo_fn}, bmx={bmx_fn}, bpx={bpx_fn}, ghb={ghb_fn}, "
        f"hdl={hdl_fn}, tchol={tchol_fn}, trig={trig_fn}, crp={crp_fn}, "
        f"biochem={biochem_fn}, glu={glu_fn}, cbc={cbc_fn}"
    )
    print(f"{cycle['label']} rows={len(out):,} non-null counts={nonnull_summary}")

    return out

    return out

In [ ]:
# rebuild all cycles for the biomarker master
df_all = pd.concat([load_cycle(c) for c in CYCLES], ignore_index=True)
df_all["cycle_start_year"] = df_all["CYCLE"].map(cycle_start_year_from_label).astype("Int64")

display(df_all.groupby("CYCLE")[["LBXSAL", "LBXSCR", "LBXSAPSI"]].count())

print("\nAll-cycle biomarker master shape:", df_all.shape)
print("All-cycle counts:")
display(df_all["CYCLE"].value_counts().sort_index())

# survival-eligible subset is explicit and separate
df_surv_base = df_all[df_all["CYCLE"].isin(VALID_SURVIVAL_CYCLES)].copy()

print("\nSurvival-eligible base shape:", df_surv_base.shape)
print("Survival-eligible cycle counts:")
display(df_surv_base["CYCLE"].value_counts().sort_index())


In [ ]:
CORE_13 = [
    "BMXBMI", "BMXWAIST", "BPXSY1", "BPXDI1",
    "LBXGH", "LBDHDD", "LBXTC", "LBXTR",
    "LBXSAL", "LBXSCR", "LBXSAPSI",
    "LBXWBCSI", "LBXLYPCT"
]

EXTENDED_16 = CORE_13 + [
    "LBXGLU", "LBXMCV", "LBXRDW"
]

print("CORE_13 complete-case counts by cycle:")
print(df_all.groupby("CYCLE")[CORE_13].apply(lambda x: x.notna().all(axis=1).sum()))

print("\nEXTENDED_16 complete-case counts by cycle:")
print(df_all.groupby("CYCLE")[EXTENDED_16].apply(lambda x: x.notna().all(axis=1).sum()))

In [ ]:
# save all-cycle biomarker master (working artifact)
df_all.to_parquet(BIOMARKER_MASTER_PATH, index=False)
print("Saved biomarker master to:", BIOMARKER_MASTER_PATH)


In [ ]:
import pandas as pd
import requests

def parse_mortality_dat(content: bytes) -> pd.DataFrame:
    text = content.decode("latin-1", errors="ignore")
    lines = text.splitlines()

    rows = []
    for line in lines:
        if not line.strip():
            continue

        seqn_str = line[0:5].strip() if len(line) >= 5 else ""
        eligstat_str = line[14:15].strip() if len(line) >= 15 else ""
        mortstat_str = line[15:16].strip() if len(line) >= 16 else ""
        ucod_str = line[16:19].strip() if len(line) >= 19 else ""
        diabetes_str = line[19:20].strip() if len(line) >= 20 else ""
        hyperten_str = line[20:21].strip() if len(line) >= 21 else ""
        permth_int_str = line[42:45].strip() if len(line) >= 45 else ""
        permth_exm_str = line[45:48].strip() if len(line) >= 48 else ""

        if not seqn_str.isdigit():
            continue

        rows.append({
            "SEQN": pd.to_numeric(seqn_str, errors="coerce"),
            "ELIGSTAT": pd.to_numeric(eligstat_str, errors="coerce"),
            "MORTSTAT": pd.to_numeric(mortstat_str, errors="coerce"),
            "UCOD_LEADING": pd.to_numeric(ucod_str, errors="coerce"),
            "DIABETES": pd.to_numeric(diabetes_str, errors="coerce"),
            "HYPERTEN": pd.to_numeric(hyperten_str, errors="coerce"),
            "PERMTH_INT": pd.to_numeric(permth_int_str, errors="coerce"),
            "PERMTH_EXM": pd.to_numeric(permth_exm_str, errors="coerce"),
        })

    df = pd.DataFrame(rows)
    df["SEQN"] = pd.to_numeric(df["SEQN"], errors="coerce").astype("Int64")
    return df


mort_base = "https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality"

mort_frames = []
for cycle in SURVIVAL_CYCLES:
    url = f"{mort_base}/{cycle['mort']}"
    print(f"Loading mortality: {cycle['label']}")
    r = requests.get(url, timeout=60, allow_redirects=True)
    r.raise_for_status()

    df_mort = parse_mortality_dat(r.content)
    df_mort["CYCLE_SRC"] = cycle["label"]
    mort_frames.append(df_mort)

mort_all = pd.concat(mort_frames, ignore_index=True)
mort_all = mort_all.drop_duplicates(subset=["SEQN"]).copy()
mort_all["SEQN"] = pd.to_numeric(mort_all["SEQN"], errors="coerce").astype("Int64")

print("\nMortality file shape:", mort_all.shape)
print("Mortality unique SEQN:", mort_all["SEQN"].nunique())
print("Mortality cycle counts:")
display(mort_all["CYCLE_SRC"].value_counts().sort_index())


In [ ]:
# save verified mortality file
mort_all.to_parquet(MORTALITY_PATH, index=False)
print("Saved mortality file to:", MORTALITY_PATH)


In [ ]:

# ============================================================
# MERGE SURVIVAL-ELIGIBLE BASE WITH VERIFIED MORTALITY
# ============================================================

df_surv_base[ID_COL] = pd.to_numeric(df_surv_base[ID_COL], errors="coerce").astype("Int64")
mort_all[ID_COL] = pd.to_numeric(mort_all[ID_COL], errors="coerce").astype("Int64")

mort_keep = [c for c in [ID_COL, "ELIGSTAT", EVENT_COL, TIME_COL, "PERMTH_EXM", "CYCLE_SRC"] if c in mort_all.columns]

merged = df_surv_base.merge(
    mort_all[mort_keep],
    on=ID_COL,
    how="left",
    validate="one_to_one"
).copy()

merged["cycle_start_year"] = merged[CYCLE_COL].map(cycle_start_year_from_label).astype("Int64")

print("Merged shape:", merged.shape)
print("Merged columns:")
print(merged.columns.tolist())
print("\nMerged cycle counts:")
display(merged[CYCLE_COL].value_counts().sort_index())


In [ ]:

# ============================================================
# RAW POST-MERGE COVERAGE AUDIT (informational only)
# ============================================================

assert "merged" in globals(), "merged dataframe not found — run previous cells"

coverage_raw = (
    merged.groupby(CYCLE_COL)[[EVENT_COL, TIME_COL]]
    .apply(lambda x: x.notna().mean())
)

print("Raw post-merge mortality coverage by cycle:")
display(coverage_raw)


In [ ]:

# ============================================================
# BUILD FINAL SURVIVAL COHORT
# ============================================================

survival_df = merged.copy()

if "ELIGSTAT" in survival_df.columns:
    before = len(survival_df)
    survival_df = survival_df[survival_df["ELIGSTAT"] == 1].copy()
    print(f"Filtered ELIGSTAT==1: {before:,} -> {len(survival_df):,}")

survival_df = survival_df[
    survival_df[TIME_COL].notna() & survival_df[EVENT_COL].notna()
].copy()

print("Final survival cohort shape:", survival_df.shape)

coverage_final = (
    survival_df.groupby(CYCLE_COL)[[EVENT_COL, TIME_COL]]
    .apply(lambda x: x.notna().mean())
)

print("Final survival cohort coverage by cycle:")
display(coverage_final)

assert survival_df[ID_COL].is_unique, "Duplicate SEQN in final survival cohort"
assert survival_df[TIME_COL].notna().all(), "Missing PERMTH_INT in final survival cohort"
assert survival_df[EVENT_COL].notna().all(), "Missing MORTSTAT in final survival cohort"
assert set(survival_df[CYCLE_COL].unique()).issubset(set(VALID_SURVIVAL_CYCLES)), "Unexpected cycle present in final survival cohort"

print("Final survival cohort:")
print("N:", len(survival_df))
print("Deaths:", int(survival_df[EVENT_COL].sum()))
print("Event rate:", survival_df[EVENT_COL].mean())
print("Cycles:", sorted(survival_df[CYCLE_COL].unique()))
print(survival_df[TIME_COL].describe())
print(survival_df[EVENT_COL].value_counts())


In [ ]:

# ============================================================
# SAVE FULL SURVIVAL MASTER (FOR BASELINE RISK)
# ============================================================

survival_df.to_parquet(SURVIVAL_MASTER_PATH, index=False)

print("Saved survival master to:", SURVIVAL_MASTER_PATH)
print("Shape:", survival_df.shape)
print("Deaths:", int(survival_df[EVENT_COL].sum()))


In [ ]:

# build extended survival-ready dataset from audited survival cohort
df_ext16_surv = survival_df.dropna(
    subset=EXTENDED_16
).copy()

df_ext16_surv["cycle_start_year"] = (
    df_ext16_surv[CYCLE_COL].map(cycle_start_year_from_label).astype("Int64")
)

assert df_ext16_surv[ID_COL].is_unique, "SEQN is not unique after survival filtering"
assert df_ext16_surv[TIME_COL].notna().all(), "Missing survival time detected"
assert df_ext16_surv[EVENT_COL].notna().all(), "Missing event indicator detected"
assert set(df_ext16_surv[CYCLE_COL].unique()).issubset(set(VALID_SURVIVAL_CYCLES)), "Unexpected cycle present"

print(df_ext16_surv.shape)
print("Deaths:", float(df_ext16_surv[EVENT_COL].sum()))
print("Cycle counts:")
display(df_ext16_surv[CYCLE_COL].value_counts().sort_index())


In [ ]:

# save extended survival dataset
df_ext16_surv.to_parquet(EXTENDED_MODEL_PATH, index=False)

print("Saved extended survival dataset to:", EXTENDED_MODEL_PATH)


In [ ]:

# split by time
train = df_ext16_surv[df_ext16_surv[CYCLE_COL].isin([
    "1999-2000", "2001-2002", "2003-2004",
    "2005-2006", "2007-2008", "2009-2010"
])].copy()

test = df_ext16_surv[df_ext16_surv[CYCLE_COL].isin([
    "2011-2012", "2013-2014", "2015-2016"
])].copy()

print(train.shape, test.shape)


In [ ]:
!pip install lifelines

In [ ]:
#quick baseline model (sanity check)
from lifelines import CoxPHFitter

cph = CoxPHFitter()

cph.fit(
    train[[TIME_COL, EVENT_COL] + EXTENDED_16],
    duration_col=TIME_COL,
    event_col=EVENT_COL
)

cph.print_summary()

In [ ]:
#evaluate on test
test["risk_score"] = cph.predict_partial_hazard(test)

from lifelines.utils import concordance_index

c_index = concordance_index(
    test[TIME_COL],
    -test["risk_score"],
    test[EVENT_COL]
)

print("Test C-index:", c_index)

In [ ]:

print("Merged columns:")
print(merged.columns.tolist())

print("\nMerged cycle counts:")
display(merged[CYCLE_COL].value_counts().sort_index())

print("\nExtended survival dataset cycle counts:")
display(df_ext16_surv[CYCLE_COL].value_counts().sort_index())

print("\nAny missing survival values in merged?")
print(
    merged.groupby(CYCLE_COL)[[TIME_COL, EVENT_COL]]
    .apply(lambda x: x.isna().sum())
    .sort_index()
)


In [ ]:
print("Final survival cohort:")
print("N:", len(survival_df))
print("Deaths:", int(survival_df["MORTSTAT"].sum()))
print("Event rate:", survival_df["MORTSTAT"].mean())
print("Cycles:", sorted(survival_df["CYCLE"].unique()))

In [ ]:
print(survival_df["PERMTH_INT"].describe())
print(survival_df["MORTSTAT"].value_counts())